# Oxta — Stack Smoke via train_curriculum.py

**Estratégia revisada:** o caminho direto (`engine.train_text` em loop) quebrou com `Tensor allocation failed` porque `load_model('')` deixa o tokenizer vazio. Em vez de lutar com isso, usamos o pipeline de produção: **`train_curriculum.py`** via subprocess.

Esse pipeline:
- Usa o bundle `distillation_bundle_v11` (tokenizer + dados de currículo já validados)
- Profile auto-detectado pelo `colab_bootstrap` baseado no GPU (T4 → `hybrid_v11_colab_t4_80m`, 80M com Mamba2 + Attention + MoE)
- BF16 Tensor Cores ON via env var
- Sai loss por fase do treino — basta parsear

Pra ser **smoke** (não treino real), passamos `--override-phase-steps 15` — total ~90 steps em 6 fases, ~3-5 min de treino real (+ ~5 min boot).

**Critério único:** loss inicial cai entre fases. Se cair, stack funcional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

REPO_OWNER = 'vitorGgC569'
REPO_NAME = 'reimagined'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')
DRIVE_ROOT = Path('/content/drive/MyDrive/nsos_v11')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        clone_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('[auth] GITHUB_TOKEN OK')
except Exception:
    pass

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    clone_url, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'set-url',
                    'origin', clone_url], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'colab'))
os.environ['NSOS_PARAM_SCALE'] = '80m'
os.environ['NSOS_MIXED_PRECISION'] = 'bf16'

import colab_bootstrap
info = colab_bootstrap.setup_environment(verbose=True)
print(f'\nBuild OK on {info["gpu_name"]}')
print(f'profile auto-detected: {info["profile"]}')
print(f'bundle_ready:          {info["bundle_ready"]}')
if not info['bundle_ready']:
    print('\nAVISO: bundle não cached — primeira sessão pode demorar 30-60min só pra')
    print('fazer o build do dataset. Se preferir mais rápido, pega uma sessão depois')
    print('que a v11 já rodou pelo menos uma vez (cache fica em Drive).')

In [ ]:
# Localiza o bundle. Bootstrap já extraiu pra esse path se bundle_ready=True.
V11_BUNDLE = REPO_ROOT / 'OXN/nsos/scripts/distillation_bundle_v11'

if not V11_BUNDLE.exists():
    raise RuntimeError(
        f'Bundle não encontrado em {V11_BUNDLE}.\n'
        f'Provavelmente precisa rodar train_v11.ipynb uma vez antes pra cachear o '
        f'distillation_bundle_v11.zip no Drive — sem isso o smoke não consegue iniciar.'
    )

bundle_size = sum(f.stat().st_size for f in V11_BUNDLE.rglob('*') if f.is_file())
n_files = sum(1 for f in V11_BUNDLE.rglob('*') if f.is_file())
print(f'Bundle: {V11_BUNDLE}')
print(f'  {n_files} arquivos, {bundle_size / 1024**2:.1f} MB')
print(f'\nArquivos top-level:')
for f in sorted(V11_BUNDLE.iterdir())[:15]:
    sz = f.stat().st_size / 1024 if f.is_file() else 0
    print(f'  {f.name}: {sz:.0f} KB')

## Treino smoke via train_curriculum.py

Override `phase-steps=15` faz cada fase rodar só 15 steps em vez do default (32-128). Total ~90 steps em 6 fases. Roda em ~3-5 min de treino + ~2-3 min de boot.

In [ ]:
import time
RUN_NAME = f'smoke_{info["profile"].split("_")[-1]}_{time.strftime("%Y%m%d_%H%M")}'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['PYTHONPATH'] = str(Path(info['ext_so']).parent)
env['NSOS_BUILD_DIR'] = info['build_dir']
env['NSOS_TRAIN_CHUNK_SIZE'] = '0'      # full batch em T4
env['PYTHONUNBUFFERED'] = '1'
env['NSOS_MIXED_PRECISION'] = 'bf16'

cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'OXN/nsos/scripts/train_curriculum.py'),
    '--profile', info['profile'],
    '--bundle-dir', str(V11_BUNDLE),
    '--build-dir', info['build_dir'],
    '--run-dir', str(RUN_DIR),
    '--device', 'gpu',
    '--override-phase-steps', '15',     # ← SMOKE: drasticamente reduzido
    '--phase-eval-samples', '4',         # ← eval rápido por fase
    '--phase-eval-mode', 'fast',
    '--checkpoint-every-steps', '0',     # smoke não precisa checkpoint mid-phase
    '--global-suite-samples-per-phase', '2',
]

# Streaming de output line-by-line (mesmo padrão do v11)
if shutil.which('stdbuf'):
    cmd = ['stdbuf', '-oL', '-eL'] + cmd

print(f'profile:   {info["profile"]}')
print(f'bundle:    {V11_BUNDLE}')
print(f'run_dir:   {RUN_DIR}')
print(f'override-phase-steps: 15 (smoke — pra treino real é 32-128 default)')
print(f'\nRodando train_curriculum.py...\n')

# Captura output em memória pra parsing posterior
captured_lines = []
t_start = time.time()

proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        captured_lines.append(line.rstrip())
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    print('\n[smoke] interrompido pelo usuário')
    proc.terminate()
    proc.wait(timeout=10)

total_min = (time.time() - t_start) / 60
print(f'\n\n[smoke] subprocess terminou em {total_min:.1f}min com exit code {proc.returncode}')


## Parse loss + verdict

In [ ]:
import re
import matplotlib.pyplot as plt

# Padrões comuns que train_curriculum imprime:
#   '[phase1_algorithms] step 5/15 loss=2.345 ...'
#   '[phase X] avg_loss=2.123 ...'
#   '... loss: 1.23 ...'

loss_pattern = re.compile(r'loss[\s=:]+([0-9]+\.[0-9]+)', re.IGNORECASE)
phase_pattern = re.compile(r'\[(phase\d+_\w+|instruction_polish|global)\]', re.IGNORECASE)

losses = []
phase_markers = []  # (line_idx, phase_name)
current_phase = None

for i, line in enumerate(captured_lines):
    pm = phase_pattern.search(line)
    if pm:
        new_phase = pm.group(1)
        if new_phase != current_phase:
            phase_markers.append((len(losses), new_phase))
            current_phase = new_phase
    lm = loss_pattern.search(line)
    if lm:
        try:
            val = float(lm.group(1))
            if 0.001 < val < 50:  # filtra valores absurdos (não é loss)
                losses.append(val)
        except ValueError:
            pass

print(f'Capturadas {len(losses)} medições de loss em {len(captured_lines)} linhas')
print(f'Fases identificadas: {[p[1] for p in phase_markers]}')

if not losses:
    print('\n❌ Não consegui extrair loss do output.')
    print('Manda as últimas 30 linhas do log de cima pra eu ajustar o regex.')
else:
    plt.figure(figsize=(11, 4))
    plt.plot(losses, marker='o', markersize=3)
    for idx, phase in phase_markers:
        plt.axvline(idx, color='gray', linestyle=':', alpha=0.4)
        plt.text(idx, max(losses) * 0.95, phase[:15], rotation=90, fontsize=7, alpha=0.7)
    plt.xlabel('measurement #')
    plt.ylabel('loss')
    plt.title(f'Oxta smoke via train_curriculum — {info["profile"]}, {len(losses)} loss samples')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/oxta_smoke_curriculum_loss.png', dpi=110)
    plt.show()

    # Verdict
    loss_init = sum(losses[:5]) / min(5, len(losses))
    loss_final = sum(losses[-5:]) / min(5, len(losses))
    drop_pct = ((loss_init - loss_final) / loss_init * 100) if loss_init > 0 else 0
    print(f'\n{"="*70}')
    if proc.returncode != 0:
        print(f'❌ Subprocess terminou com erro (exit {proc.returncode})')
    elif drop_pct < 10:
        print(f'⚠️ INSUFICIENTE: loss caiu apenas {drop_pct:.1f}% ({loss_init:.2f} → {loss_final:.2f})')
        print(f'   Pode ser smoke curto demais — aumenta --override-phase-steps pra 30+')
    else:
        print(f'✅ PASSOU: loss caiu {drop_pct:.1f}% ({loss_init:.2f} → {loss_final:.2f})')
        print(f'   Stack inteiro do Oxta (Mamba2 + Attention + MoE + BF16) está funcional.')
    print(f'{"="*70}')
    print(f'  loss inicial: {loss_init:.3f}')
    print(f'  loss final:   {loss_final:.3f}')
    print(f'  loss min:     {min(losses):.3f}')
    print(f'  loss max:     {max(losses):.3f}')
    print(f'  fases:        {len(phase_markers)}')
    print(f'  exit code:    {proc.returncode}')
    print(f'  run_dir:      {RUN_DIR}')

## Próximos passos

**Se ✅ PASSOU:** stack do Oxta tá vivo. Próximos:
1. Treino real: o mesmo notebook **sem** `--override-phase-steps` (deixa defaults, ~4-8h em T4)
2. Ou ainda mais rápido: usa `train_v11.ipynb` que já tá tunado pro pipeline completo

**Se ⚠️ INSUFICIENTE:** aumenta `--override-phase-steps` pra 30-50 e re-roda. Smoke de 15 pode ser pouco pra ver descida.

**Se ❌ FALHOU:** manda as últimas 30 linhas do log + exit code. Possíveis causas:
- Bundle corrompido (re-extrair)
- Profile incompatível com binding atual
- CUDA OOM (reduzir profile pra `pilot`)